# GRPO Chess Colab Runner

Run the full searchless pipeline in Colab:
1. Pretrain
2. Distill (warm-start from pretrain)
3. GRPO (warm-start from distill)

This notebook loads committed smoke/full configs from `src/configs/` for end-to-end execution.


In [ ]:
#@title Runtime Parameters
REPO_URL = "https://github.com/noamdwc/grpo_chess.git"  #@param {type:"string"}
REPO_REF = "feature/lightning_training"  #@param {type:"string"}
RUN_PROFILE = "full"  #@param ["smoke", "full"]
USE_DRIVE = True  #@param {type:"boolean"}
HF_CACHE_ROOT = "/content/drive/MyDrive/data/grpo-chess/hf_cache"  #@param {type:"string"}
USE_WANDB = True  #@param {type:"boolean"}
WANDB_API_KEY = ""  #@param {type:"string"}
RUN_PRETRAIN = True  #@param {type:"boolean"}
RUN_DISTILL = True  #@param {type:"boolean"}
RUN_GRPO = True  #@param {type:"boolean"}


In [ ]:
# Setup workspace
import os
import sys
from pathlib import Path

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

hf_cache_root = Path(HF_CACHE_ROOT if USE_DRIVE else '/content/hf_cache')
hf_cache_root.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(hf_cache_root)
os.environ['HF_DATASETS_CACHE'] = str(hf_cache_root / 'datasets')
os.environ['TRANSFORMERS_CACHE'] = str(hf_cache_root / 'transformers')
print(f"HF cache root: {hf_cache_root}")

if USE_WANDB:
    wandb_key = WANDB_API_KEY.strip()
    if not wandb_key:
        try:
            from google.colab import userdata
            wandb_key = (userdata.get('WANDB_API_KEY') or userdata.get('WANDB_KEY') or '').strip()
        except Exception:
            wandb_key = ''

    if not wandb_key:
        raise RuntimeError(
            'USE_WANDB=True but no key was found. Set WANDB_API_KEY in runtime params or Colab Secrets.'
        )

    os.environ['WANDB_API_KEY'] = wandb_key
    os.environ['WANDB_KEY'] = wandb_key
    print('WandB key configured via environment; logger login happens during training.')
else:
    os.environ['WANDB_DISABLED'] = 'true'
    print('WandB disabled for this notebook run.')

!rm -rf /content/grpo_chess
!git clone {REPO_URL} /content/grpo_chess
%cd /content/grpo_chess
!git fetch --all --tags
!git checkout {REPO_REF}
!git submodule update --init --recursive

if '/content/grpo_chess' not in sys.path:
    sys.path.append('/content/grpo_chess')


In [ ]:
# Install dependencies
from pathlib import Path
import os
import signal

deps_ready = Path('/tmp/grpo_colab_deps_ready')
if not deps_ready.exists():
    %pip install -q --upgrade pip setuptools wheel
    # Avoid local numpy pin in requirements.txt; we install a coherent binary stack below.
    !grep -vE '^numpy==' requirements.txt > /tmp/requirements-colab.txt
    %pip install -q -r /tmp/requirements-colab.txt
    # Force-reinstall core numeric packages together to avoid ABI mismatches.
    %pip install -q --force-reinstall --no-cache-dir \
        "numpy==2.1.3" "scipy==1.14.1" "pandas==2.2.2" "pyarrow==18.1.0" \
        "requests==2.32.4" "urllib3<=2.5.0" "jedi>=0.19.1"
    !apt-get -qq update
    !apt-get -qq install -y stockfish
    !which stockfish || true
    deps_ready.write_text('ok')
    print('Dependencies installed. Restarting runtime to load fresh binary modules...')
    os.kill(os.getpid(), signal.SIGKILL)
else:
    !which stockfish || true
    !python -V
    import numpy, scipy, pandas, pyarrow
    print('numpy', numpy.__version__)
    print('scipy', scipy.__version__)
    print('pandas', pandas.__version__)
    print('pyarrow', pyarrow.__version__)


## Select Pipeline Configs

This notebook uses committed profile configs from `src/configs/`.
- `smoke`: `*_colab_e2e_smoke.yaml`
- `full`: `*_colab_e2e_full.yaml`
- Paths/checkpoint handoff are encoded in those YAML files.


In [ ]:
from pathlib import Path
import yaml

repo = Path('/content/grpo_chess')
config_dir = repo / 'src' / 'configs'

run_profile = RUN_PROFILE.strip().lower()
if run_profile not in {'smoke', 'full'}:
    raise ValueError(f'RUN_PROFILE must be one of ["smoke", "full"], got: {RUN_PROFILE}')

pretrain_config_name = f'pretrain_colab_e2e_{run_profile}.yaml'
distill_config_name = f'distill_colab_e2e_{run_profile}.yaml'
grpo_config_name = f'grpo_colab_e2e_{run_profile}.yaml'

pretrain_config_path = config_dir / pretrain_config_name
distill_config_path = config_dir / distill_config_name
grpo_config_path = config_dir / grpo_config_name

for config_path in [pretrain_config_path, distill_config_path, grpo_config_path]:
    if not config_path.exists():
        raise FileNotFoundError(f'Missing committed config: {config_path}')

with pretrain_config_path.open('r') as f:
    pretrain_yaml = yaml.safe_load(f)
with distill_config_path.open('r') as f:
    distill_yaml = yaml.safe_load(f)
with grpo_config_path.open('r') as f:
    grpo_yaml = yaml.safe_load(f)

pretrain_ckpt_dir = Path(pretrain_yaml['pretrain']['checkpoint_dir'])
distill_ckpt_dir = Path(distill_yaml['distill']['checkpoint_dir'])
grpo_ckpt_dir = Path(grpo_yaml['training']['checkpoint_dir'])
distill_data_dir = Path(distill_yaml['dataset']['data_dir'])
pretrain_processed_cache_dir = Path(pretrain_yaml['dataset']['cache_path'])
hf_datasets_cache_dir = Path(pretrain_yaml['dataset']['hf_cache_dir'])
artifact_root = pretrain_ckpt_dir.parents[1]

for d in [
    pretrain_ckpt_dir,
    distill_ckpt_dir,
    grpo_ckpt_dir,
    distill_data_dir,
    pretrain_processed_cache_dir,
    hf_datasets_cache_dir,
]:
    d.mkdir(parents=True, exist_ok=True)

print('Run profile:', run_profile)
print('Committed configs:')
print('-', pretrain_config_name)
print('-', distill_config_name)
print('-', grpo_config_name)
print('Artifact root:', artifact_root)
print('HF datasets cache:', hf_datasets_cache_dir)


## Run Pipeline Stages


In [ ]:
# Stage 1: Pretrain
if RUN_PRETRAIN:
    import torch
    from src.pretrain.pretrain import load_pretrain_config, train as pretrain_train

    torch.set_float32_matmul_precision('high')
    pretrain_config, pretrain_dataset_config, pretrain_transformer_config, pretrain_eval_cfg, pretrain_stockfish_cfg, pretrain_policy_cfg = load_pretrain_config(
        pretrain_config_name,
    )
    print(f'Pretrain config: {pretrain_config_name}')
    print(f'Pretrain checkpoint dir: {pretrain_config.checkpoint_dir}')
    pretrain_final_path = pretrain_train(
        pretrain_config,
        pretrain_dataset_config,
        pretrain_transformer_config,
        pretrain_eval_cfg,
        pretrain_stockfish_cfg,
        pretrain_policy_cfg,
    )
else:
    print('Skipping pretrain stage')


In [ ]:
# Stage 2: Distill (warm-start from pretrain_final.pt)
if RUN_DISTILL:
    from src.distill.distill import (
        ensure_distill_dataset,
        load_distill_config,
        train as distill_train,
    )

    distill_config, distill_dataset_config, distill_transformer_config, distill_eval_cfg, distill_stockfish_cfg, distill_policy_cfg = load_distill_config(
        distill_config_name,
    )
    print(f'Distill config: {distill_config_name}')
    print(f'Distill checkpoint dir: {distill_config.checkpoint_dir}')
    ensure_distill_dataset(distill_config_name, distill_dataset_config)
    distill_final_path = distill_train(
        distill_config,
        distill_dataset_config,
        distill_transformer_config,
        distill_eval_cfg,
        distill_stockfish_cfg,
        distill_policy_cfg,
    )
else:
    print('Skipping distill stage')


In [ ]:
# Stage 3: GRPO (warm-start from distill_final.pt)
if RUN_GRPO:
    import torch
    from src.train_self_play import train as grpo_train

    torch.set_float32_matmul_precision('high')
    print(f'GRPO config: {grpo_config_name}')
    grpo_train(
        config_path=grpo_config_name,
        dataloader_kwargs={'num_workers': 0},
    )
else:
    print('Skipping GRPO stage')


In [ ]:
# Verify expected artifacts
from pathlib import Path

checks = {
    'pretrain_final': pretrain_ckpt_dir / 'pretrain_final.pt',
    'distill_final': distill_ckpt_dir / 'distill_final.pt',
    'grpo_dir': grpo_ckpt_dir,
}
for name, path in checks.items():
    print(f'{name}:', 'OK' if path.exists() else f'MISSING ({path})')


## Notes

- This notebook loads committed profile configs from `src/configs/`.
- Profile selection uses `RUN_PROFILE` and picks `*_colab_e2e_smoke.yaml` or `*_colab_e2e_full.yaml`.
- Paths/checkpoint handoff are defined directly in the committed YAML files.
- Set `RUN_PROFILE="full"` for full runs, or `RUN_PROFILE="smoke"` for quick checks.
- Full pretrain config is L4-safe (`batch_size=512`, `accumulate_grad_batches=4`, `precision="bf16-mixed"`).
- This project is searchless; this notebook does not introduce tree-search/MCTS steps.
